# Synthetic Taxi Data — 再現実験 (Google Colab GPU版)

論文「A Systematic Evaluation of Generative Models on Tabular Transportation Data (arXiv:2502.08856)」の再現実験

**事前設定**: ランタイム → ランタイムのタイプを変更 → GPU を選択

## 1. 環境構築

In [ ]:
# GPU確認
!nvidia-smi

In [ ]:
# リポジトリ取得
!git clone https://github.com/chengenw/transportation.git
%cd transportation

In [ ]:
# 依存パッケージインストール
!pip install -q \
    torch torchvision torchaudio \
    pandas numpy scipy scikit-learn matplotlib seaborn \
    tqdm pyarrow \
    copulas sdv sdmetrics deepecho rdt \
    category-encoders graphviz cloudpickle \
    POT termcolor wget icecream \
    ml-collections tomli-w \
    psutil tensorboard

# rtdl は依存衝突があるため --no-deps
!pip install -q rtdl --no-deps

# TensorFlow (STaSy用、不要ならスキップ可)
!pip install -q tensorflow

In [ ]:
# setGPU をコメントアウト (Colab環境では不要)
!sed -i 's/^import setGPU/# import setGPU/' tab_main.py

# GPU利用確認
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Green データセット — 本実験 (N=40,000)

### 2.1 GaussianCopula (統計ベース、高速)

In [ ]:
!python tab_main.py \
    --dataset green --nums 40000 \
    --methods GaussianCopula \
    --outfile colab_gc_green \
    --nexp 3 --ngen 5

### 2.2 CTGAN

In [ ]:
!python tab_main.py \
    --dataset green --nums 40000 \
    --methods CTGAN \
    --outfile colab_ctgan_green \
    --nexp 3 --ngen 5

### 2.3 TVAE

In [ ]:
!python tab_main.py \
    --dataset green --nums 40000 \
    --methods TVAE \
    --outfile colab_tvae_green \
    --nexp 3 --ngen 5

### 2.4 CTABGAN

In [ ]:
!python tab_main.py \
    --dataset green --nums 40000 \
    --methods CTABGAN \
    --outfile colab_ctabgan_green \
    --nexp 3 --ngen 5

### 2.5 TabDDPM

In [ ]:
!python tab_main.py \
    --dataset green --nums 40000 \
    --methods TabDDPM \
    --outfile colab_tabddpm_green \
    --nexp 3 --ngen 5

### 2.6 STaSy (OOMリスクあり、失敗時はスキップ)

In [ ]:
!python tab_main.py \
    --dataset green --nums 40000 \
    --methods STaSy \
    --outfile colab_stasy_green \
    --nexp 3 --ngen 5

## 3. Zone データセット — グラフ構造評価 (N=40,000)

In [ ]:
models_zone = ['GaussianCopula', 'CTGAN', 'TVAE', 'TabDDPM']

for model in models_zone:
    print(f'\n{"="*60}')
    print(f'Zone experiment: {model}')
    print(f'{"="*60}')
    !python tab_main.py \
        --dataset zone --nums 40000 \
        --methods {model} \
        --outfile colab_{model.lower()}_zone \
        --nexp 3 --ngen 5

## 4. 結果確認

In [ ]:
import pandas as pd
import glob

# Green 結果の読み込み
columns = [
    'num', 'model',
    'dwn_tr_tr', 'dwn_tr_syn', 'dwn_tr_te', 'dwn_syn_syn', 'dwn_syn_tr', 'dwn_syn_te',
    'G_tr_te', 'G_tr_syn', 'G_te_syn',
    'sdv_tr_te', 'sdv_tr_syn', 'sdv_te_syn',
    'w1_tr_te', 'w1_tr_syn', 'w1_te_syn',
    'cov_tr_te', 'cov_tr_syn', 'cov_te_syn',
    'dcr_rs', 'dcr_hs', 'rDCR', 'perc', 'dcr_rr', 'dcr_ss'
]

result_files = sorted(glob.glob('output/colab_*_green_*.csv'))
print('=== Green Dataset Results ===')
for f in result_files:
    df = pd.read_csv(f, header=0, skipinitialspace=True)
    row = df.iloc[-1]  # 最終行
    model = str(row.iloc[1]).strip()
    print(f'\n--- {model} ---')
    print(f'  Downstream R² (syn→te): {str(row.iloc[7]).strip()}')
    print(f'  SDV Score (te_syn):      {str(row.iloc[13]).strip()}')
    print(f'  Wasserstein (te_syn):    {str(row.iloc[16]).strip()}')
    print(f'  Coverage (te_syn):       {str(row.iloc[19]).strip()}')
    print(f'  rDCR:                    {str(row.iloc[22]).strip()}')

In [ ]:
# 実行速度の確認
speed = pd.read_csv('output/speed.csv', skipinitialspace=True)
print('=== Training Time (minutes) ===')
print(speed.groupby('method')['minutes'].agg(['mean', 'std', 'count']).round(2))

In [ ]:
# 結果ファイルをダウンロード
from google.colab import files
import shutil

shutil.make_archive('/content/colab_results', 'zip', 'output')
files.download('/content/colab_results.zip')
print('結果ファイルをダウンロードしました')